# **Automated Comprehensibility Assessment Tool (A-CAT)**

In this script, we will show you how to get a comprehensibility assessment on a single audio file.

Please follow this step-by-step tutorial, by pressing the play buttons next to the code chunks. Please execute the code in the exact same order as presented in this script. After pressing the button, please wait until a green tick (when code executed correctly) or red exclamation mark (when code encountered some errors) appears next to the button before moving forward. Pressing too many buttons at the same time might cause the script to crash.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1gJyDbe9IpJwBIzgbx4Q18RZHbZBkhBLe)


## Step 1a: Import packages
The code chunk below imports all packages necessary to run our automated speech assessment. Some of the packages are readily available via Google Colab, others need to be installed.

If any of them shows as missing (i.e., you see some errors displayed below the code section), please install them with !pip command in the code chunk below (Step 1b) and come back to this chunk to import them once installed.

In [25]:
from os.path import join, expanduser
import parselmouth
from parselmouth.praat import call, run_file
from os.path import join
from praatio import textgrid
import csv
from csv import writer
from csv import reader
import numpy as np
import pandas as pd
from glob import glob
import math
import io
from IPython.display import display, HTML

## Step 1b: Install packages (optional)
If you don't see any errors (usually displayed in red), please skip this step.

If you noticed some errors in Step 1a, you might need to install additional packages. Install additional packages only if you see the following error: "ModuleNotFoundError". 

This error indicates that the packages you are trying to import are not currently available. Any packages that are not found or were not previously installed can be installed with the !pip command in the section below. You can find appropriate !pip commands for all packages you can use in Python here: https://pypi.org/

In [24]:
!pip install praat-parselmouth
!pip install praatio

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 83.5 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 6.1 MB/s eta 0:00:00


## Step 2: Define working directory
For the script to work correctly, it needs to have access to the relevant files, functions and audio recording for analysis.

First, download the folder called 'ACAT' and upload it to your Google Drive. Remember to maintain the structure of the original folder.

Then, mount your Google Drive by giving necessary permissions, so that the Google Colab platform can read its contents. There will be a pop-up window asking you to do so, so please follow the instructions on your screen to give access.

In [26]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


If you want to analyze your own voice, please record a 30-seconds WAV file (please make sure you speak for 30 seconds, otherwise the analysis won't be reliable). When recording your audio, please follow the instructions below to guarantee the best signal quality.

---
Voice recording tips:
- Please make sure that you sit in a quiet room and make sure you won't be interrupted
- Close the window to minimize noise from outside
- If you hear some ambient noise, nice idea is to cover yourself and a microphone with a blanket to record your voice - blanket shields you from background noise
- Switch off any sound notifications and put your phone aside (inluding irregular sounds like this will distort your score)
- Speak directly to the microphone and do not cover your mouth
- Please try not to breath loudly into the microphone
- Speak for 30 seconds (uploading shorter or longer files will give you unreliable results)
---

Below we explicitly specify where our audio file is located (i.e., inside the folder called 'examples'). We need to make sure that that folder is inside the folder called 'ACAT' that contains the folders with Praat scripts and your WAV file you wish to analyze. For now, let's work with the example audio file from our 'examples' folder.

Please make sure that this folder contains only one audio file.

Please check the folder location on your local Google Drive and update accordingly.

In [27]:
# Please note that Windows and Mac use \ vs / so please change the format accordingly
root_dir = expanduser('/content/drive/MyDrive/ACAT/examples/')
filetype = '*.wav'
loc = join(root_dir, filetype)
print(loc)
# You should see below something like: '/content/drive/MyDriveACAT/examples/*.wav'

/content/drive/MyDrive/ACAT/examples/*.wav


If Google Drive is connected and ACAT folder uploaded correctly, all the contents should be visible in the folder pane on the left.

## Step 3: Extract parameters

In [28]:
# Extract syllables and acoustic features
tab1 = parselmouth.praat.run_file('drive/MyDrive/ACAT/functions/SyllableNucleiv3.praat', loc, 'None', -25, 2, 0.4, True, 'English', 1, 'Table', 'OverWriteData', True)

# Read the data from the Praat generated Table Object (annotated syllables and filled pauses)
df1 = pd.read_table(io.StringIO(parselmouth.praat.call(tab1[2], "List", False)))

# Select relevant columns and replace missing values with NANs
selected_cols = ['type', 'F0', 'F1', 'F2', 'F3']
df1 = df1[selected_cols]

df1 = df1[df1['type'] == 'syll']
df1.replace('--undefined--', np.nan, inplace=True)
df1 = df1.astype({'F0': float, 'F1': float, 'F2': float, 'F3': float})

df1_sub = pd.DataFrame({
    'meanf0': df1['F0'].mean(skipna=True), 
    'meanf1': df1['F1'].mean(skipna=True),
    'meanf2': df1['F2'].mean(skipna=True),
    'meanf3': df1['F3'].mean(skipna=True),
    
    'sdf0': df1['F0'].std(skipna=True),
    'sdf1': df1['F1'].std(skipna=True),
    'sdf2': df1['F2'].std(skipna=True),
    'sdf3': df1['F3'].std(skipna=True),
    
    'minf0': df1['F0'].min(skipna=True),
    'maxf0': df1['F0'].max(skipna=True)
}, index=[0])

df1_sub['rangef0'] = df1_sub['maxf0'] - df1_sub['minf0']
df1_sub['coeff1'] = math.log10(df1_sub['sdf1'] / df1_sub['meanf1'])
df1_sub['coeff2'] = math.log10(df1_sub['sdf2'] / df1_sub['meanf2'])
df1_sub['coeff3'] = math.log10(df1_sub['sdf3'] / df1_sub['meanf3'])
df1_sub = df1_sub[['rangef0', 'coeff1', 'coeff2', 'coeff3']]

# Extract fluency metrics computed by Praat script
# We run the same function for the second time to extract different metrics
tab2 = parselmouth.praat.run_file('drive/MyDrive/ACAT/functions/SyllableNucleiv3.praat', loc, 'None', -25, 2, 0.4, True, 'English', 1, 'Table', 'OverWriteData', False)
df2 = pd.read_table(io.StringIO(parselmouth.praat.call(tab2[0], "List", False)))
df2.columns = df2.columns.str.strip()
df2 = df2.rename(columns={'speechrate(nsyll/dur)': 'speechrate'})
df2['pauses'] = df2['npause'] + df2['nrFP']
df2 = df2[['speechrate', 'pauses']]


The final step involves reading individual timestamps of syllable annotations to compute syllable durations. Check the name of the TextGrid file in the folder containing the audio file and include it below.

In [29]:
inputFN = join('drive/MyDrive/ACAT/examples/101.auto.TextGrid') # Check file name and change here
tg = textgrid.openTextgrid(inputFN, includeEmptyIntervals=False)
tier = tg.getTier(tg.tierNames[2])


In [43]:
start_times = []
end_times = []
labels = []

for interval in tier.entries:
    start_times.append(interval[0])
    end_times.append(interval[1])
    labels.append(interval[2])


df3 = pd.DataFrame(np.column_stack([start_times, end_times, labels, np.subtract(end_times, start_times)]),
                  columns=['start', 'stop', 'type', 'diff'])
df3 = df3.astype({'start': float, 'stop': float, 'diff': float})
df3 = df3[df3['type'] == 'syll']
df3.replace('--undefined--', np.nan, inplace=True)


df_sub3 = pd.DataFrame({
    'sdsylldur': math.log10(df3['diff'].std(skipna=True))
}, index=[0])

final = pd.concat([df2, df_sub3, df1_sub], axis=1)

# Calculate comprehensibility
comp_avg = 2.138+(2.701 * final['speechrate']) + (0.015 * final['pauses'])+(-0.020 * final['rangef0'])+(3.821 * final['sdsylldur'])+(-1.414 * final['coeff1'])+(-5.549 * final['coeff2'])+(3.228* final['coeff3'])
#print(f"Your comprehensibility score is: {comp_avg.iloc[0]}")
comp_score = comp_avg.iloc[0]
comp_text = f"<h2 style='color: black;'>Your comprehensibility score is: <strong><span style='color: red;'>{comp_score}</span></strong></h2>"
display(HTML(comp_text))

# Calculate nativelikeness
native_avg = -0.537+(2.654*final['speechrate'])+(-0.001*final['pauses'])+(-0.019*final['rangef0'])+(3.170*final['sdsylldur'])+(-0.622*final['coeff1'])+(-8.016*final['coeff2'])+(3.575*final['coeff3'])
#print(f"Your nativelikeness score is: {native_avg.iloc[0]}")
native_score = native_avg.iloc[0]
native_text = f"<h2 style='color: black;'>Your nativelikeness score is: <strong><span style='color: red;'>{native_score}</span></strong></h2>"
display(HTML(native_text))

## **How to interpret the comprehensibility score**


Score  | Suggested interpretation
-------------------|------------------
2.171-3.409 | Low comprehensibility – representing inexperienced L2 learners’ speech (no immersion experience)
3.492-4.308 | Mid comprehensibility – representing moderately experienced L2 learners’ speech (LOR 1 month to 5 years)
5.378-6.147 | High comprehensibility- representing long-term L2 residents’ speech (LOR 6-18 years)
7.6 | Near-native L2 speaker (maximum)
8.714-8.926 | Native speakers of English




## **How to interpret the nativelikeness score**


Score  | Suggested interpretation
-------------------|------------------
1.445-1.955 | Low nativelikeness – representing inexperienced L2 learners’ speech (no immersion experience)
2.312-3.053 | Mid nativelikeness – representing moderately experienced L2 learners’ speech (LOR 1 month to 5 years) 
3.852-4.668 | High nativelikeness - representing long-term L2 residents’ speech (LOR 6-18 years)
6.9 | Near-native L2 speaker (maximum)
8.450-8.930 | Native speakers of English